# 01a — Train: FlanT5_Stepwise_FT_Combined

Trains the **proposed method**: Flan-T5-XL fine-tuned with LoRA on a combination
of FOLIO logical entailment data and PlanBench stepwise action records.

## Core idea

Both FOLIO and stepwise plan verification share the same logical structure:
given a set of premises, determine whether a conclusion is entailed (A) or
contradicted (B). By training on both tasks under the same prompt format,
the model acquires general logical reasoning capability from FOLIO and adapts
to planning domain vocabulary via a small number of PlanBench fewshot examples.

The model is trained on **individual action steps** — one premise→conclusion pair
per record. It never sees a full plan during training. This is the fundamental
structural difference from direct fine-tuning (01c), which trains on plan-level
binary labels and fails to learn precondition checking.

## Training data

| Source | Size | Purpose |
|---|---|---|
| FOLIO train split | 677 examples (388 A, 289 B) | Logical entailment capability |
| PlanBench fewshot stepwise | 94 step records (25 fewshot_invalid plans) | Domain adaptation |
| **Combined** | **771 records (464 A, 307 B)** | **Training** |

## Checkpoint selection

FOLIO validation F1-macro — early stopping patience 3.
The FOLIO validation set is held out from PlanBench evaluation entirely.

## Output

Saves LoRA adapter weights (~40MB) to `paper2/adapters/FlanT5_Stepwise_FT_Combined/`.

## Prerequisites

Run `00_data_preparation.ipynb` first.

## 1. Setup

DEBUG mode (cell 2) runs a quick 8-example / 2-epoch sanity check to verify
the pipeline runs without errors before committing to a full training run.
Set `DEBUG=False` for the actual training run.

Mount Google Drive, install dependencies, and log into HuggingFace
(required to download Flan-T5-XL and the FOLIO dataset).

In [ ]:
# ── DEBUG MODE ────────────────────────────────────────────────────────────────
# Set DEBUG=True to run a quick sanity check (8 examples, 2 epochs).
# This verifies the full pipeline — data loading, tokenisation, forward pass,
# metric computation, checkpoint saving — in under 2 minutes before committing
# to a full training run (~2-4 hours on A100).
# Always run DEBUG=True first. Only set DEBUG=False when you are confident.

DEBUG   = False   # Set False for full training
DEBUG_N = 8
print(f'Mode: {"DEBUG" if DEBUG else "FULL TRAINING"}')

In [ ]:
# ── RESET BEFORE FULL RUN ─────────────────────────────────────────────────────
# Run this cell BEFORE switching from DEBUG=True to DEBUG=False.
# It deletes debug checkpoints and the debug adapter so full training
# starts cleanly from the pretrained Flan-T5-XL base.
#
# DO NOT run this if you want to resume a previously interrupted full run —
# in that case leave the checkpoints in place and the trainer will resume.

import shutil, os

RESET = False   # ← set True only when switching DEBUG=True → DEBUG=False

if RESET:
    targets = [CKPT_DIR, ADAPTER_DIR, TOK_DIR]
    for t in targets:
        if os.path.exists(t):
            shutil.rmtree(t)
            print(f'Deleted: {t}')
        else:
            print(f'Not found (skip): {t}')
    print('\nReset complete. Now set DEBUG=False and run all cells.')
else:
    print('RESET=False — nothing deleted.')
    print('Set RESET=True to clear debug outputs before a full run.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes scikit-learn huggingface_hub

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via HF_TOKEN secret')
except Exception:
    from huggingface_hub import login
    login()

In [ ]:
import os, gc, json, random, warnings
import numpy as np
import torch
from collections import Counter
from dataclasses import dataclass as _dc

from datasets import load_dataset, Dataset as HFDataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,
    EarlyStoppingCallback, TrainerCallback, IntervalStrategy,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, accuracy_score, f1_score

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR    = f'{ROOT}/data'
OUT_ROOT    = f'{ROOT}/adapters/FlanT5_Stepwise_FT_Combined'
ADAPTER_DIR = os.path.join(OUT_ROOT, 'lora_adapter')
TOK_DIR     = os.path.join(OUT_ROOT, 'tokenizer')
CKPT_DIR    = os.path.join(ROOT, 'checkpoints', 'FlanT5_Stepwise_FT_Combined')
RESULTS_DIR = f'{ROOT}/results'
for d in [OUT_ROOT, CKPT_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

SW_FEWSHOT_PATH = os.path.join(DATA_DIR, 'planbench_stepwise_fewshot.jsonl')

# ── Model config ──────────────────────────────────────────────────────────────
# Flan-T5-XL: 3B parameter encoder-decoder. Pretrained on a diverse mixture of
# NLP tasks including NLI, which makes it a strong starting point for entailment.
MODEL_NAME     = 'google/flan-t5-xl'
MAX_SOURCE_LEN = 1024   # sufficient for all stepwise prompts (longest ~600 tokens)
MAX_TARGET_LEN = 4      # target is a single token: 'A' or 'B'

# ── Batch config ──────────────────────────────────────────────────────────────
# Effective batch size = PER_DEVICE_BS × GRAD_ACCUM = 2 × 8 = 16
# Gradient accumulation simulates a larger batch without exceeding GPU memory.
PER_DEVICE_BS = 2
GRAD_ACCUM    = 8

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']


use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
def cleanup(): gc.collect(); torch.cuda.empty_cache()

print(f'ROOT        : {ROOT}')
print(f'ADAPTER_DIR : {ADAPTER_DIR}')
print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'BF16        : {use_bf16}')

## 2. Prompt Builder

A single prompt template is used for both FOLIO and PlanBench records,
enabling transfer learning between the two tasks:

```
Task: Determine whether the conclusion is entailed or contradicted given the premises.
Premises:
- <premise 1>
- <premise 2>
  ...
Conclusion:
<conclusion>

Output format: Answer A if the conclusion is entailed, B if it is contradicted.
Answer:
```

For **FOLIO**: premises are natural language statements from first-order logic;
the conclusion is a hypothesis to verify.

For **PlanBench stepwise**: premises are domain action rules plus current world
state facts; the conclusion is the proposed action as a testable hypothesis
(e.g., "The red block can be picked up.").

Both produce a single output token: `A` (entailed / action executable) or
`B` (contradicted / precondition violated).

In [ ]:
def build_folio_prompt(premises, conclusion):
    """
    Unified prompt template for both FOLIO and PlanBench stepwise records.

    FOLIO example:
      Premises: - All students are people. - Alice is a student.
      Conclusion: Alice is a person.
      Answer: A

    PlanBench example:
      Premises: - You can pick up X if X is clear and hand is empty.
                - The orange block is clear. - Hand is empty.
      Conclusion: The orange block can be picked up.
      Answer: A
    """
    if isinstance(premises, (list, tuple)):
        prem_block = '\n'.join(
            '- ' + p.strip().rstrip('.') + '.'
            for p in premises if p.strip())
    else:
        prem_block = '- ' + str(premises).strip()
    return (
        'Task: Determine whether the conclusion is entailed or '
        'contradicted given the premises.\n'
        'Premises:\n' + prem_block + '\n\n'
        'Conclusion:\n' + conclusion.strip() + '\n\n'
        'Output format: Answer A if the conclusion is entailed, '
        'B if it is contradicted.\n'
        'Answer:'
    )

def normalize_pred(text):
    """
    Extract A or B from model output. Default to B (invalid/contradicted)
    if the output is empty or unrecognisable. This conservative default
    means ambiguous outputs are counted as invalid predictions.
    """
    if not text: return 'B'
    c = text.strip()[0].upper()
    return c if c in {'A', 'B'} else 'B'

print('Prompt builder: OK')

## 3. Load PlanBench Fewshot Records (Training)

Loads the 94 stepwise action records built from the 25 `fewshot_invalid` plans
by `00_data_preparation.ipynb`. Each record is one action-verification step:
domain rules + current world state as premises, the proposed action as the
conclusion, and a gold label A (executable) or B (precondition violated).

**Class imbalance:** most steps are labelled A. Invalid plans typically have
one failing step (labelled B) with all preceding steps labelled A. Goal-fail
plans contribute only A-labelled steps. This imbalance is corrected by the
weighted loss function in §6.

In [ ]:
# ── Fewshot: goes into training ──────────────────────────────────────────────
pb_fewshot_raw = [json.loads(l) for l in open(SW_FEWSHOT_PATH)]
print(f'Fewshot records loaded: {len(pb_fewshot_raw)}')
cnt = Counter(r['label'] for r in pb_fewshot_raw)
print(f'  A={cnt["A"]}  B={cnt["B"]}')
print(f'  Class ratio A:B = {cnt["A"]/max(1,cnt["B"]):.1f}:1  '
      f'(handled by weighted loss)')

## 4. Load FOLIO and Build Combined Training Set

### FOLIO dataset
FOLIO is a human-annotated natural language reasoning dataset grounded in
first-order logic. Each example has premises, a conclusion, and a label:
True (entailed → A), False (contradicted → B), or Uncertain (dropped).
After dropping Uncertain examples: 677 training, 134 validation.

FOLIO provides the **logical reasoning backbone**: the model learns to check
whether a conclusion follows from premises, independent of domain semantics —
which is exactly what stepwise action verification requires.

### Combined training set
FOLIO train (677) + PlanBench fewshot (94) are concatenated and shuffled each
epoch under the same prompt format. The ~7:1 ratio (FOLIO:PlanBench) is
intentional — FOLIO provides the base reasoning capability; PlanBench adapts
it to planning domain vocabulary.

### Class weights
`w_c = N / (2 × N_c)` computed from the combined training label distribution
to correct for A/B imbalance. Applied in the weighted loss (§6).

In [ ]:
LABEL_MAP = {'True': 'A', 'False': 'B'}
ALT       = {'true': 'True', 'false': 'False', True: 'True', False: 'False'}

def norm_label(raw):
    """Normalise FOLIO labels (True/False/true/false/bool) to A/B.
    Unknown labels are dropped (return None) — we only train on binary cases."""
    s = ALT.get(raw, str(raw).strip())
    return LABEL_MAP.get(s, None)

raw_folio = load_dataset('yale-nlp/FOLIO')
print('FOLIO raw train:', Counter(raw_folio['train']['label']))
print('FOLIO raw val  :', Counter(raw_folio['validation']['label']))

def process_folio_split(split):
    """Convert FOLIO examples to the unified prompt format.
    Drops Unknown-labelled examples (not binary)."""
    out = []
    for ex in raw_folio[split]:
        lbl = norm_label(ex['label'])
        if lbl is None: continue   # drop Unknown
        out.append({
            'user_text'   : build_folio_prompt(ex['premises'], ex['conclusion']),
            'label_letter': lbl
        })
    return out

folio_train = HFDataset.from_list(process_folio_split('train'))
folio_val   = HFDataset.from_list(process_folio_split('validation'))

# Inject PlanBench fewshot into training — same format, different domain
pb_train_ds = HFDataset.from_list([
    {'user_text': r['prompt'], 'label_letter': r['label']}
    for r in pb_fewshot_raw
])
combined_train = concatenate_datasets([folio_train, pb_train_ds])

folio_ds = DatasetDict({'train': combined_train, 'validation': folio_val})
print(f'\nCombined train : {len(combined_train)} '
      f'(FOLIO={len(folio_train)} + PlanBench_fewshot={len(pb_train_ds)})')
print(f'Validation     : {len(folio_val)} (FOLIO only — used for checkpoint selection)')

# ── Class weights ─────────────────────────────────────────────────────────────
# Computed on the combined training set. Used to penalise the majority class
# (A = entailed/valid) more lightly and the minority class (B = contradicted/invalid)
# more heavily, preventing the model from collapsing to always predicting A.
# Formula: w_c = n_total / (n_classes × n_c)
# This is the inverse-frequency weighting scheme.
all_labels = combined_train['label_letter']
n_A = all_labels.count('A'); n_B = all_labels.count('B'); n_total = n_A + n_B
w_A = n_total / (2 * n_A)
w_B = n_total / (2 * n_B)
print(f'\nClass balance  : A={n_A} ({n_A/n_total*100:.1f}%)  B={n_B} ({n_B/n_total*100:.1f}%)')
print(f'Class weights  : A={w_A:.3f}  B={w_B:.3f}')
print(f'  → B-steps penalised {w_B/w_A:.1f}x more than A-steps')

if DEBUG:
    folio_ds = DatasetDict({
        k: folio_ds[k].select(range(min(DEBUG_N, len(folio_ds[k]))))
        for k in folio_ds
    })
    print(f'[DEBUG] train={len(folio_ds["train"])} val={len(folio_ds["validation"])}')

## 5. Tokenise

Flan-T5 is an encoder-decoder model. The input prompt is tokenised for the
encoder; the target label (`A` or `B`) is the decoder target sequence.

- `MAX_SOURCE_LEN = 1024` tokens: sufficient for all stepwise prompts
  (longest Logistics prompts reach ~600 tokens)
- `MAX_TARGET_LEN = 4` tokens: target is always a single token (`A` or `B`)

Label token IDs for `A` and `B` are extracted from the vocabulary and passed
to the weighted loss function.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    """
    Tokenise input prompts and target labels for Seq2Seq training.
    - Input: full prompt string → encoder input_ids + attention_mask
    - Target: 'A' or 'B' → decoder labels (pad tokens → -100 for loss masking)
    """
    model_inputs = tokenizer(
        batch['user_text'],
        max_length=MAX_SOURCE_LEN, truncation=True, padding=False)
    labels_enc = tokenizer(
        text_target=batch['label_letter'],
        max_length=MAX_TARGET_LEN, truncation=True, padding=False)
    pad_id = tokenizer.pad_token_id
    # Replace padding token ids in labels with -100 so they are ignored by loss
    model_inputs['labels'] = [
        [(t if t != pad_id else -100) for t in seq]
        for seq in labels_enc['input_ids']
    ]
    return model_inputs

tokenized = folio_ds.map(
    tokenize_batch, batched=True,
    remove_columns=folio_ds['train'].column_names)
print('Tokenised dataset:', tokenized)

# Check token ids for A and B — used in weighted loss
_a_id = tokenizer.encode('A', add_special_tokens=False)[0]
_b_id = tokenizer.encode('B', add_special_tokens=False)[0]
print(f'Token ID: A={_a_id}  B={_b_id}')

## 6. Model, LoRA, and Weighted Loss

### Base model
Flan-T5-XL (3B parameters, `google/flan-t5-xl`): an encoder-decoder model
instruction-tuned on a diverse collection of NLP tasks including natural language
inference. Loaded in BFloat16 to reduce GPU memory.

### LoRA (Low-Rank Adaptation)
LoRA inserts trainable low-rank matrices into the attention projections,
reducing trainable parameters from 3B to ~9.4M (0.33%).

| Hyperparameter | Value | Meaning |
|---|---|---|
| `r` | 8 | Rank of the low-rank decomposition |
| `lora_alpha` | 16 | Scaling factor (effective scale = alpha/r = 2) |
| `lora_dropout` | 0.05 | Dropout for regularisation |
| `target_modules` | q, k, v, o | All attention projections in encoder and decoder |

### Weighted cross-entropy loss
Without weighting, the model predicts A for everything on imbalanced data.
Weights `w_c = N / (2 × N_c)` give higher penalty to misclassifying the
minority class (B). Applied only to the A and B token positions in the
vocabulary via a custom `Seq2SeqTrainer` subclass.

### Gradient checkpointing
Recomputes activations during the backward pass to reduce peak GPU memory.
Required to fit Flan-T5-XL on a single GPU.

In [ ]:
cleanup()
base = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if use_bf16 else torch.float32,
    device_map='auto',
)
# Gradient checkpointing: recompute activations during backward to save GPU memory
base.gradient_checkpointing_enable()
if hasattr(base, 'enable_input_require_grads'):
    base.enable_input_require_grads()

# Apply LoRA: only ~9.4M parameters will be trained
model = get_peft_model(base, LoraConfig(
    r=8,                              # low-rank dimension
    lora_alpha=16,                    # scaling: effective LR = alpha/r × lr = 2 × 2e-4
    lora_dropout=0.05,                # regularisation
    bias='none',                      # don't train bias terms
    task_type=TaskType.SEQ_2_SEQ_LM,  # encoder-decoder
    target_modules=['q', 'k', 'v', 'o'],  # all attention projections
))
model.print_trainable_parameters()

A_ID = tokenizer.encode('A', add_special_tokens=False)[0]
B_ID = tokenizer.encode('B', add_special_tokens=False)[0]
print(f'Token IDs: A={A_ID}  B={B_ID}')

if hasattr(model, 'generation_config'):
    model.generation_config.max_length = MAX_TARGET_LEN + 1

class WeightedSeq2SeqTrainer(Seq2SeqTrainer):
    """
    Extends HuggingFace Seq2SeqTrainer with class-weighted cross-entropy loss.

    Standard CE loss treats all vocab tokens equally.
    Here we upweight the loss on B-token predictions (invalid/contradicted)
    because B-label steps are the minority class and are the hardest to learn —
    they are the only steps where the model must identify a precondition violation.

    Implementation: build a weight vector of shape (vocab_size,) filled with 1.0,
    then set w[A_ID] and w[B_ID] to the computed inverse-frequency weights.
    CrossEntropyLoss(weight=w) then scales the loss at positions where the target
    is A or B accordingly. All other vocab positions (padding, EOS, etc.) keep
    weight 1.0 but are masked to -100 in the labels so they don't contribute.
    """
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            w = torch.ones(
                self.model.config.vocab_size,
                device=next(self.model.parameters()).device)
            w[A_ID] = class_weights[0]  # w_A (lower — majority class)
            w[B_ID] = class_weights[1]  # w_B (higher — minority class)
            self.loss_fct = torch.nn.CrossEntropyLoss(weight=w, ignore_index=-100)
        else:
            self.loss_fct = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.loss_fct is None:
            return super().compute_loss(model, inputs, return_outputs, **kwargs)
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits  # shape: (batch, seq_len, vocab_size)
        # Flatten to (batch×seq_len, vocab_size) for cross-entropy
        loss = self.loss_fct(
            logits.view(-1, logits.size(-1)),
            labels.view(-1).to(logits.device))
        return (loss, outputs) if return_outputs else loss

print(f'BF16: {use_bf16}')

## 7. Training Configuration

### Checkpoint selection
Best checkpoint selected by FOLIO validation F1-macro. FOLIO validation is
used rather than PlanBench performance to avoid any leakage from the test pool
into training decisions.

### Early stopping
Patience = 3 evaluation steps on FOLIO val F1-macro.

### Hyperparameters

| Parameter | Value |
|---|---|
| Learning rate | 2 × 10⁻⁴ |
| LR scheduler | Cosine decay |
| Per-device batch size | 2 |
| Gradient accumulation steps | 8 |
| Effective batch size | 16 |
| Warmup steps | 65 |
| Precision | BFloat16 |
| Loss | Weighted cross-entropy |
| Seed | 42 |

In [ ]:
@_dc
class _Log: last_train_loss: float = float('nan')
_log = _Log()

class _TableLogger(TrainerCallback):
    """Captures training loss for display purposes."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            _log.last_train_loss = float(logs['loss'])


def compute_metrics(eval_pred):
    """
    Compute accuracy and F1-macro on the FOLIO validation set.
    F1-macro is the checkpoint selection metric (metric_for_best_model).

    F1-macro treats both classes equally regardless of frequency,
    which is appropriate here because both Entailed and Contradicted
    are equally important to predict correctly.
    """
    pred_ids, label_ids = eval_pred
    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids  = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    preds = [normalize_pred(t) for t in pred_texts]
    golds = [normalize_pred(t) for t in gold_texts]
    accuracy = sum(p == g for p, g in zip(preds, golds)) / max(1, len(golds))
    f1_macro = f1_score(
        [1 if g == 'A' else 0 for g in golds],
        [1 if p == 'A' else 0 for p in preds],
        average='macro', zero_division=0)
    # Per-class accuracy for diagnosis
    for letter, name in [('A', 'entailed'), ('B', 'contradicted')]:
        idxs = [i for i, g in enumerate(golds) if g == letter]
        if idxs:
            cls_acc = sum(preds[i] == letter for i in idxs) / len(idxs)
            print(f'  acc_{name}: {cls_acc:.3f}  ({len(idxs)} examples)')
    return {'accuracy': accuracy, 'f1_macro': f1_macro}

training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,                    # HuggingFace checkpoints here
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM, # effective batch = 2 × 8 = 16
    num_train_epochs=2 if DEBUG else 50,    # 50 epoch ceiling; early stopping triggers first
    learning_rate=2e-4,
    warmup_steps=65,                        # ~1 epoch warmup
    lr_scheduler_type='cosine',             # cosine decay after warmup
    weight_decay=0.0,
    eval_strategy=IntervalStrategy.STEPS,
    eval_steps=5 if DEBUG else 200,
    save_steps=5 if DEBUG else 200,
    save_total_limit=2,                     # keep only 2 checkpoints to save disk
    load_best_model_at_end=True,            # restore best checkpoint after training
    metric_for_best_model='eval_f1_macro',  # FOLIO val F1-macro for selection
    greater_is_better=True,
    predict_with_generate=True,             # use model.generate() for eval decoding
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=1 if DEBUG else 50,
    report_to='none',
    fp16=False, bf16=bool(use_bf16),
    dataloader_num_workers=2, seed=SEED,
)

callbacks = [
    _TableLogger(),
    EarlyStoppingCallback(early_stopping_patience=3),  # stop if FOLIO F1 flat for 3 evals
]

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True)

trainer = WeightedSeq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
    class_weights=(w_A, w_B),  # passed to WeightedSeq2SeqTrainer
)

print(f'Train records  : {len(tokenized["train"])}')
print(f'Val records    : {len(tokenized["validation"])}')
print(f'Effective batch: {PER_DEVICE_BS} × {GRAD_ACCUM} = {PER_DEVICE_BS*GRAD_ACCUM}')
print(f'Max epochs     : {training_args.num_train_epochs}')
print(f'Early stopping : patience=5 on FOLIO val F1-macro')
cleanup()
trainer.train()
best = trainer.state.best_metric
print(f'\nBest FOLIO val F1-macro: {best:.4f}' if best else 'Best: N/A')

## 8. FOLIO Validation Report

Runs inference on the full FOLIO validation set (134 examples) with the
best checkpoint and prints a classification report.

This confirms the best checkpoint has genuinely learned logical entailment
on the held-out FOLIO validation set. The F1-macro reported here is the
metric used for checkpoint selection.

In [ ]:
pred_out   = trainer.predict(tokenized['validation'])
pred_texts = tokenizer.batch_decode(pred_out.predictions, skip_special_tokens=True)
lab_ids    = np.where(
    pred_out.label_ids != -100,
    pred_out.label_ids, tokenizer.pad_token_id)
gold_texts = tokenizer.batch_decode(lab_ids, skip_special_tokens=True)
pf = [normalize_pred(t) for t in pred_texts]
gf = [normalize_pred(t) for t in gold_texts]
lmap = {'A': 'Entailed', 'B': 'Contradicted'}
print('=== FOLIO BINARY VALIDATION REPORT ===')
print(classification_report(
    [lmap.get(g, '?') for g in gf],
    [lmap.get(p, '?') for p in pf],
    labels=['Entailed', 'Contradicted'], zero_division=0))

## 9. PlanBench Monitor

Not reported here. A monitor set drawn from the test pool was used for
informational logging during training only and never influenced checkpoint
selection. It is excluded from this notebook to avoid any appearance of
test-set usage during training.

In [ ]:
# Monitor section removed — PlanBench test pool not used during training.

## 10. Save Adapter

Saves the LoRA adapter weights (~40MB) and tokenizer to
`paper2/adapters/FlanT5_Stepwise_FT_Combined/`.

Only the adapter delta weights are saved — the base Flan-T5-XL model
is downloaded from HuggingFace at inference time.

To load the trained model in `02_eval_flant5.ipynb`:
```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
base      = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-xl')
model     = PeftModel.from_pretrained(base, ADAPTER_DIR)
tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
```

In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(TOK_DIR)
print(f'Adapter   → {ADAPTER_DIR}')
print(f'Tokenizer → {TOK_DIR}')
print()
print('=== TRAINING COMPLETE ===')
print(f'Condition      : FlanT5_Stepwise_FT_Combined')
print(f'Training data  : FOLIO + PlanBench fewshot stepwise')
print(f'Best FOLIO val F1-macro: {trainer.state.best_metric:.4f}'
      if trainer.state.best_metric else 'Best metric: N/A')
print(f'\nNext step: run 01b_train_stepwise_ablations.ipynb')